In [47]:
# Spacy for NLP tasks
import spacy
from spacy import displacy
from spacy.pipeline import EntityRuler
from spacy.language import Language
from spacy.tokens import Doc

# Subprocess for downloading spacy models
import sys
import subprocess

# Zero-shot NLI model for custom entity recognition
from transformers import pipeline as hf_pipeline

# Download the spaCy English model (small — fast, ~12 MB)
# en_core_web_sm is trained on OntoNotes 5 (news + web)
subprocess.check_call([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'])

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 2.8 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


0

# Phase 4 — Named Entity Recognition

**Goal**: Find people, orgs, dates, money amounts etc. in the text. (src/pipeline/stage3_ner.py)

### How spaCy works

When you call `nlp(text)`, your text passes through a **pipeline** of components:

```
raw text
   │
   ▼
┌─────────────┐   Splits text into Token objects by whitespace/punctuation rules
│  tokenizer  │
└──────┬──────┘
       │
       ▼
┌─────────────┐   Assigns part-of-speech tags (NOUN, VERB, ADJ…)
│   tagger    │
└──────┬──────┘
       │
       ▼
┌─────────────┐   Assigns fine-grained dependency labels (nsubj, dobj…)
│   parser    │
└──────┬──────┘
       │
       ▼
┌─────────────┐   Assigns named entity spans (PERSON, ORG, GPE…)
│     ner     │
└──────┬──────┘
       │
       ▼
   Doc object  ← everything lives here
```

### The Doc object

A `Doc` is a sequence of `Token` objects. Every token stores:

| Attribute | Description | Example |
|-----------|-------------|--------|
| `token.text` | Original string | `"NASA"` |
| `token.lemma_` | Base (dictionary) form | `"nasa"` |
| `token.pos_` | Coarse POS tag | `"PROPN"` |
| `token.tag_` | Fine-grained POS tag | `"NNP"` |
| `token.dep_` | Dependency label | `"nsubj"` |
| `token.is_stop` | Is it a stopword? | `False` |
| `token.ent_type_` | Named entity type | `"ORG"` |

### How spaCy tokenises

spaCy uses a **rule-based tokeniser** that applies:
1. **Prefix rules** — strips leading punctuation (e.g. `(NASA` → `(` + `NASA`)
2. **Suffix rules** — strips trailing punctuation (e.g. `said.` → `said` + `.`)
3. **Infix rules** — splits on hyphens, slashes (e.g. `well-known` → `well` + `-` + `known`)
4. **Exception rules** — keeps contractions and abbreviations intact (e.g. `U.S.` stays as one token)

This is more linguistically aware than simply splitting on whitespace.

**Part-of-speech tagging** assigns a grammatical category (noun, verb, adjective, etc.) to each token.  
These tags are critical for:
- NER (proper nouns are likely entity candidates)
- Information extraction (verbs signal relationships)
- Text summarisation (content words vs. function words)

### Two tag sets in spaCy

| Attribute | Tag set | Example |
|-----------|---------|--------|
| `token.pos_` | Universal POS (coarse, 17 tags) | `NOUN`, `VERB`, `PROPN` |
| `token.tag_` | Penn Treebank (fine-grained, 36+ tags) | `NN`, `VBD`, `NNP` |

Common coarse tags: `NOUN`, `PROPN` (proper noun), `VERB`, `ADJ`, `ADV`, `DET`, `PUNCT`, `NUM`, `ADP` (preposition)

**Dependency parsing** builds a tree that shows *grammatical relationships* between words.  
Each token (except the root verb) has an *arc* pointing to its syntactic head.

Example tree for `"NASA launched the rocket."`:  
```
         launched          ← ROOT (main verb)
        /        \
      NASA        rocket   ← nsubj (subject), dobj (direct object)
                  |
                 the       ← det (determiner)
```

### Why it matters for NLP

Dependency structure lets us:
- Find the subject of a verb (who did what?)
- Extract (subject, verb, object) triples for knowledge graphs
- Identify which adjective modifies which noun

### What is NER?

**Named Entity Recognition** is the task of locating and classifying *named entities* in text  
into predefined categories such as person names, organisations, locations, dates, and monetary values.

Example:  
```
"Apple acquired Intel's modem business for $1 billion in October 2019."
                                                                        
 Apple     → ORG   (organisation)
 Intel     → ORG
 $1 billion→ MONEY
 October 2019 → DATE
```

### spaCy's entity types (OntoNotes 5 label set)

| Label | Meaning |
|-------|--------|
| `PERSON` | People, including fictional |
| `ORG` | Companies, agencies, institutions |
| `GPE` | Countries, cities, states (Geopolitical Entity) |
| `LOC` | Non-GPE locations (mountains, bodies of water) |
| `DATE` | Absolute or relative dates |
| `TIME` | Times smaller than a day |
| `MONEY` | Monetary values including currency |
| `PERCENT` | Percentage |
| `NORP` | Nationalities, religious/political groups |
| `PRODUCT` | Objects, vehicles, foods |
| `EVENT` | Named events (wars, elections, etc.) |
| `LAW` | Named laws and legal documents |

### How NER models work

spaCy uses a **transition-based** NER system.  
The model reads tokens left-to-right and decides for each token whether it:
- **B**egins a new entity span
- Is **I**nside an ongoing entity span
- Is **O**utside any entity

This BIO (or IOB2) tagging scheme is the standard format for sequence labelling.

### Why add rule-based matching?

spaCy's statistical NER model is trained on news text and is good at general entity types.  
However, it **misses domain-specific entities** such as:
- Custom product codes or invoice numbers (`INV-2024-001`)
- Stock ticker symbols (`$AAPL`, `MSFT`)
- Structured identifiers (ISIN codes, regulatory IDs)

The **EntityRuler** component lets you define patterns using:
1. **Exact string matches** — `{"lower": "nasa"}`
2. **Regex patterns** — `{"TEXT": {"REGEX": "\\$[A-Z]{2,5}"}}`
3. **Token attribute combos** — `{"POS": "PROPN", "DEP": "nsubj"}`

Patterns are checked **before** (or after) the statistical NER, giving you full control.

In [48]:
# Load spacy model at module level so it's not reloaded on every call
nlp = spacy.load("en_core_web_sm")

print('Pipeline components:', nlp.pipe_names)
# Expected output: ['tok2vec', 'tagger', 'parser', 'senter', 'ner', 'attribute_ruler', 'lemmatizer']

print()
print('Supported entity types:')
print(nlp.get_pipe('ner').labels)

Pipeline components: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']

Supported entity types:
('CARDINAL', 'DATE', 'EVENT', 'FAC', 'GPE', 'LANGUAGE', 'LAW', 'LOC', 'MONEY', 'NORP', 'ORDINAL', 'ORG', 'PERCENT', 'PERSON', 'PRODUCT', 'QUANTITY', 'TIME', 'WORK_OF_ART')


In [49]:
# EntityRuler for custom patterns
ruler = nlp.add_pipe("entity_ruler", before="ner", config={"overwrite_ents": True})

# ── Define domain-specific patterns ───────────────────────────────────
# Each pattern is a dict with 'label' (entity type) and 'pattern' (matching rule).
patterns = [
    # Invoice numbers: match a single token like 'INV-2024-0451'
    # REGEX: 'INV-' followed by 4 digits, dash, 4 digits
    {'label': 'INVOICE_ID',
     'pattern': [{'TEXT': {'REGEX': 'INV-\d{4}-\d{4}'}}]},
    # Indian Rupee amounts: 'Rs.' followed by any number-like token
    # LIKE_NUM is True for '89,000' or '12.5' — handles comma-formatted numbers
    {'label': 'MONEY',
     'pattern': [{"TEXT": {"REGEX": "\\$[\\d,]+(\\.\\d{2})?"}}]},
    # GST numbers: 15-character alphanumeric Indian tax identifier
    # Pattern: 2 digits + 5 UPPERCASE letters + 4 digits + 1 letter + 1 alphanumeric + 'Z' + 1 alphanumeric
    {'label': 'GST_NO',
     'pattern': [{'TEXT': {'REGEX': '[0-9]{2}[A-Z]{5}[0-9]{4}[A-Z]{1}[1-9A-Z]{1}Z[0-9A-Z]{1}'}}]},
    # Indian cities: exact lower-case match using IN (list membership check)
    {'label': 'CITY_IN',
     'pattern': [{'LOWER': {'IN': ['mumbai', 'delhi', 'bengaluru', 'chennai',
                                    'hyderabad', 'pune', 'kolkata', 'ahmedabad']}}]},
    # Document type keywords
    {'label': 'DOC_TYPE',
     'pattern': [{'LOWER': {'IN': ['invoice', 'purchase order', 'contract',
                                    'agreement', 'nda', 'memorandum']}}]},
    # Fiscal quarter references: Q3 FY2024, Q1 FY2025, etc.
    {'label': 'FISCAL_PERIOD',
     'pattern': [{'TEXT': {'REGEX': 'Q[1-4]'}}, {'TEXT': {'REGEX': 'FY\d{4}'}}]},
    # Email addresses: match tokens that look like emails using LIKE_EMAIL
    {"label": "EMAIL", 
     "pattern": [{"LIKE_EMAIL": True}]}
]

ruler.add_patterns(patterns)

<>:10: SyntaxWarning: invalid escape sequence '\d'
<>:29: SyntaxWarning: invalid escape sequence '\d'
<>:10: SyntaxWarning: invalid escape sequence '\d'
<>:29: SyntaxWarning: invalid escape sequence '\d'
/var/folders/hq/v_7wt_d51sn37zqs9vlsf41c0000gn/T/ipykernel_43708/901762018.py:10: SyntaxWarning: invalid escape sequence '\d'
  'pattern': [{'TEXT': {'REGEX': 'INV-\d{4}-\d{4}'}}]},
/var/folders/hq/v_7wt_d51sn37zqs9vlsf41c0000gn/T/ipykernel_43708/901762018.py:29: SyntaxWarning: invalid escape sequence '\d'
  'pattern': [{'TEXT': {'REGEX': 'Q[1-4]'}}, {'TEXT': {'REGEX': 'FY\d{4}'}}]},


In [50]:
def spacy_ner_extraction(text: str, nlp_model: spacy.Language) -> list[dict]:
    """
    Extract named entities from text using spaCy.
    
    Args:
        text (str): Input text to analyze.
        nlp_model (spacy.Language): The loaded spaCy model.
    
    Returns:
        list[dict]: A list of dictionaries, each containing 'text', 'label', 'start_char', and 'end_char' for each entity.
    """
    # Process the text through the spaCy pipeline
    doc = nlp_model(text)
    # Initialize an empty list to hold extracted entities
    entities = []

    # Extract entities from doc object
    for ent in doc.ents:
        entities.append({
            'text': ent.text,
            'label': ent.label_,
            'start_char': ent.start_char, # Where the entity starts in the original text
            'end_char': ent.end_char # Where the entity ends in the original text
        })
    return entities

In [51]:
# Example usuage using text from sample PDF document
sample_text = """Contact Kevin William
London Walthamstow Fulbourne
Road 10 E17 4EG Data Scientist / Machine Learning Engineer
07874759550 (Mobile) Greater London, England, United Kingdom
kjw007@icloud.com
www.linkedin.com/in/kevin-william- Summary
b2b720195 (LinkedIn)
I’m a data-driven professional with a strong foundation in data
Top Skills science, artificial intelligence, and software engineering, supported
by a degree in Computational Chemistry (MChem, University of
Analytical Skills
Manchester). I’m passionate about applying analytical thinking,
Statistical Modeling
coding, and automation to help businesses make smarter, faster,
Extract, Transform, Load (ETL)
and more confident decisions.
Certifications
My technical skill set includes Python, MATLAB, and TypeScript with
Build Deep Learning Models with
TensorFlow Skill Path experience across frameworks and tools such as TensorFlow, scikit-
Learn Data Analysis with Pandas learn, Pandas, React, and Node.js. I have hands-on expertise in
Course
machine learning, NLP, predictive modeling, and data visualization,
along with experience in building scalable web applications and
analytical dashboards.
I enjoy leveraging AI and machine learning to uncover insights,
optimize processes, and drive innovation within organizations. My
approach blends technical depth with creative problem-solving —
turning complex data challenges into clear, actionable solutions that
deliver measurable business impact.
Driven by curiosity and continuous learning, I thrive at the
intersection of technology, data, and decision-making, and I’m
always looking for opportunities to apply my skills to help businesses
evolve through intelligent, data-centric strategies.
https://github.com/kjw009
Experience
Information Tech Consultants
Data Scientist
March 2026 - Present (2 months)
London Area, United Kingdom
Page 1 of 2Sika
Data Scientist / Quality Engineer
October 2022 - October 2025 (3 years 1 month)
Welwyn Garden City, England, United Kingdom
The Cultural Me
Machine Learning Engineer
November 2021 - May 2022 (7 months)
London
Education
The University of Manchester
Master's degree, Chemistry · (2017 - 2021)
Page 2 of 2"""

print("Extracted Entities:")
for entity in spacy_ner_extraction(sample_text.lower(), nlp):
    print(f"Entity Text: {entity['text']}, Label: {entity['label']}")

Extracted Entities:
Entity Text: kevin william
london, Label: PERSON
Entity Text: 10, Label: CARDINAL
Entity Text: 4eg, Label: ORDINAL
Entity Text: 07874759550, Label: DATE
Entity Text: london, Label: GPE
Entity Text: england, Label: GPE
Entity Text: united kingdom, Label: GPE
Entity Text: kjw007@icloud.com, Label: EMAIL
Entity Text: b2b720195, Label: PERSON
Entity Text: pandas, Label: ORG
Entity Text: march 2026, Label: DATE
Entity Text: 2 months, Label: DATE
Entity Text: london, Label: GPE
Entity Text: united kingdom, Label: GPE
Entity Text: 1, Label: CARDINAL
Entity Text: 2sika, Label: CARDINAL
Entity Text: october 2022 - october 2025, Label: DATE
Entity Text: 3 years 1 month, Label: DATE
Entity Text: england, Label: GPE
Entity Text: united kingdom, Label: GPE
Entity Text: november 2021 - may 2022, Label: DATE
Entity Text: 7 months, Label: DATE
Entity Text: london, Label: GPE
Entity Text: 2017 - 2021, Label: DATE
Entity Text: 2, Label: CARDINAL
Entity Text: 2, Label: CARDINAL


## Document Classifier — custom spaCy pipeline component

We add a custom component **after** the NER step so it can read the entities that spaCy has already found.

The component does two things independently, then combines them:

1. **Rule-based scoring** — counts how many domain keywords appear in the text for each category, then normalises the count to `[0, 1]` by dividing by the total keywords in that category. Fast and interpretable.
2. **Zero-shot scoring** — feeds the text to a transformer model trained on natural language inference (NLI). The model scores each candidate label without any task-specific training. Slower but handles wording the keyword list doesn't cover.

Because both scores are now on the same `0–1` scale we can compare them directly and pick the more confident result.

```
                     doc.ents  (already set by NER)
                          │
                          ▼
            ┌─────────────────────────┐
            │   document_classifier   │
            │  ┌───────────────────┐  │
            │  │  rule-based score │  │  keyword match count / vocab size → 0–1
            │  └────────┬──────────┘  │
            │           │  compare    │
            │  ┌────────▼──────────┐  │
            │  │  zero-shot score  │  │  NLI model confidence → already 0–1
            │  └───────────────────┘  │
            │           │             │
            │    winner assigned to   │
            │  doc._.doc_type         │
            │  doc._.doc_score        │
            └─────────────────────────┘
```

In [52]:
# ── Load zero-shot model once at module level ──────────────────────────
# Loading inside the component would re-initialise the model on every nlp() call.
# cross-encoder/nli-MiniLM2-L6-H768 is a lightweight NLI model (~117 MB).
# It's faster than the popular facebook/bart-large-mnli (~1.6 GB) and
# accurate enough for coarse document-type classification.
print("Loading zero-shot classifier...")
zero_shot_clf = hf_pipeline(
    "zero-shot-classification",
    model="cross-encoder/nli-MiniLM2-L6-H768",
)
print("Ready.")

Loading zero-shot classifier...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 9339.26it/s]
RobertaForSequenceClassification LOAD REPORT from: cross-encoder/nli-MiniLM2-L6-H768
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Ready.


In [53]:
# ── Register custom Doc attributes ────────────────────────────────────
if not Doc.has_extension("doc_type"):
    Doc.set_extension("doc_type", default="unknown")
if not Doc.has_extension("doc_score"):
    Doc.set_extension("doc_score", default=0.0)

# ── Keyword vocabulary per document type ──────────────────────────────
DOC_RULES = {
    "invoice":    ["invoice", "inv-", "payment due", "amount payable", "bill to", "total"],
    "contract":   ["agreement", "hereby", "terms and conditions", "clause", "signed", "parties"],
    "complaint":  ["complaint", "refund", "damaged", "issue", "disappointed", "urgent"],
    "medical":    ["patient", "hospital", "admitted", "discharged", "diagnosis", "prescription"],
    "news":       ["reported", "quarter", "profit", "growth", "announced", "market"],
    "regulatory": ["compliance", "regulation", "non-compliance", "audit", "penalty", "breach"],
    "cv":         ["experience", "education", "skills", "university", "degree", "employment"],
}

def document_classifier(doc):
    """
    Custom spaCy component: classifies the document type using rule-based
    keyword matching combined with zero-shot NLI classification.

    Writes results to:
        doc._.doc_type  (str)   — winning label e.g. 'invoice'
        doc._.doc_score (float) — confidence in [0, 1]
    """
    text_lower = doc.text.lower()

    # ── Rule-based scoring ─────────────────────────────────────────────
    # Normalise match count by vocab size so the score is in [0, 1].
    # Example: 3 matches out of 6 keywords → score = 0.5
    rule_scores = {
        label: sum(1 for kw in kws if kw in text_lower) / len(kws)
        for label, kws in DOC_RULES.items()
    }
    rule_best_label = max(rule_scores, key=rule_scores.get)
    rule_best_score = rule_scores[rule_best_label]

    # ── Zero-shot scoring ──────────────────────────────────────────────
    # Truncate to 512 chars — NLI models have a token limit and long texts
    # slow inference without meaningfully improving doc-type accuracy.
    zs_result = zero_shot_clf(text_lower[:512], candidate_labels=list(DOC_RULES.keys()))
    zs_best_label = zs_result["labels"][0]
    zs_best_score = zs_result["scores"][0]

    # ── Combine the two signals ────────────────────────────────────────
    # Both scores are now on the same [0, 1] scale so we can compare directly.
    if rule_best_score == 0.0:
        # No keyword evidence — defer entirely to the NLI model
        doc._.doc_type  = zs_best_label
        doc._.doc_score = round(zs_best_score, 3)
    elif rule_best_label == zs_best_label:
        # Both signals agree — average for a blended confidence
        doc._.doc_type  = rule_best_label
        doc._.doc_score = round((rule_best_score + zs_best_score) / 2, 3)
    elif zs_best_score > rule_best_score:
        # Disagree; zero-shot is more confident
        doc._.doc_type  = zs_best_label
        doc._.doc_score = round(zs_best_score, 3)
    else:
        # Disagree; rule-based is more confident
        doc._.doc_type  = rule_best_label
        doc._.doc_score = round(rule_best_score, 3)

    return doc


# ── Register the component ─────────────────────────────────────────────
# Language.component() doesn't support force=True when called as a function.
# has_factory() is the correct re-run guard — skip registration if already done.
if not Language.has_factory("document_classifier"):
    Language.component("document_classifier", func=document_classifier)

# add_pipe would raise an error if called twice, so guard with has_pipe.
if not nlp.has_pipe("document_classifier"):
    nlp.add_pipe("document_classifier", last=True)

print("Pipeline components:", nlp.pipe_names)

Pipeline components: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'entity_ruler', 'ner', 'document_classifier']


In [54]:
# ── Test the full pipeline on both sample documents ───────────────────
# Run each text through the complete pipeline:
#   tokenizer → tagger → parser → entity_ruler → ner → document_classifier
# The result is a Doc object with .ents populated and ._.doc_type set.

test_docs = {
    "CV": sample_text,
    "Invoice": """INVOICE
Invoice No: INV-2024-0451
Date: March 15, 2024
Bill To: GlobalTech Ltd
Address: Mumbai, India
Description          Amount
Software Development Rs 10,000
Consulting Services  Rs 2,450
TOTAL: Rs 12,450
GST @ 18%: Rs 2,241
""",
}

for doc_name, text in test_docs.items():
    doc = nlp(text)

    print(f"=== {doc_name} ===")
    print(f"  Predicted type : {doc._.doc_type}")
    print(f"  Confidence     : {doc._.doc_score}")
    print(f"  Entities found :")
    for ent in doc.ents:
        print(f"    [{ent.label_:12s}]  {ent.text!r}")
    print()

=== CV ===
  Predicted type : unknown
  Confidence     : 0
  Entities found :
    [PERSON      ]  'Kevin William'
    [ORG         ]  'London Walthamstow Fulbourne\nRoad'
    [CARDINAL    ]  '10'
    [ORG         ]  'Data Scientist / Machine Learning Engineer\n07874759550'
    [GPE         ]  'England'
    [GPE         ]  'United Kingdom'
    [EMAIL       ]  'kjw007@icloud.com'
    [PERSON      ]  'Top Skills'
    [ORG         ]  'Computational Chemistry'
    [ORG         ]  'MChem'
    [ORG         ]  'University of\nAnalytical Skills\nManchester'
    [ORG         ]  'Statistical Modeling'
    [ORG         ]  'Extract, Transform, Load (ETL'
    [ORG         ]  'Python'
    [ORG         ]  'MATLAB'
    [ORG         ]  'TypeScript'
    [WORK_OF_ART ]  'Build Deep Learning Models'
    [PERSON      ]  'Skill Path'
    [ORG         ]  'TensorFlow'
    [ORG         ]  'Learn Data Analysis'
    [PERSON      ]  'Pandas'
    [ORG         ]  'Pandas'
    [GPE         ]  'React'
    [GPE        

## Final wrapper — `run_ner_pipeline`

This function is what `src/pipeline/stage3_ner.py` will call.  
It takes raw text, runs it through the full spaCy pipeline, and returns a structured dict ready for the next stage.

In [55]:
def run_ner_pipeline(text: str) -> dict:
    """
    Run the full spaCy NER + document classification pipeline on raw text.

    Args:
        text (str): Raw document text (from OCR or direct extraction).

    Returns:
        dict with keys:
            'entities'  — list of {text, label, start_char, end_char}
            'doc_type'  — predicted document category e.g. 'invoice'
            'doc_score' — confidence score in [0, 1]
    """
    doc = nlp(text)

    entities = [
        {
            "text":       ent.text,
            "label":      ent.label_,
            "start_char": ent.start_char,
            "end_char":   ent.end_char,
        }
        for ent in doc.ents
    ]

    return {
        "entities":  entities,
        "doc_type":  doc._.doc_type,
        "doc_score": doc._.doc_score,
    }


# ── Quick smoke test ───────────────────────────────────────────────────
result = run_ner_pipeline(sample_text)
print("doc_type :", result["doc_type"])
print("doc_score:", result["doc_score"])
print("entities :", len(result["entities"]), "found")
for e in result["entities"][:5]:   # preview first 5
    print(f"  {e['label']:12s}  {e['text']!r}")

doc_type : unknown
doc_score: 0
entities : 44 found
  PERSON        'Kevin William'
  ORG           'London Walthamstow Fulbourne\nRoad'
  CARDINAL      '10'
  ORG           'Data Scientist / Machine Learning Engineer\n07874759550'
  GPE           'England'
